In [ ]:
!pip install docling langchain langchain-community langchain-huggingface langchain-pinecone pinecone-client sentence-transformers pypdf

In [ ]:
!pip install langchain==0.2.16 langchain-community==0.2.16

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.pdf'):
            print(os.path.join(root, file))

/content/drive/MyDrive/Karim_Reda_Ahmed_CV.pdf
/content/drive/MyDrive/voxmed_ai/8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy.pdf
/content/drive/MyDrive/  neural nets and apps graduate cse asu/toaz.info-pattern-classification-by-richard-o-duda-david-g-stork-peter-ehart-pr_6e9cde5bfc5e1ae22433fb169ec3d10a.pdf


In [ ]:
import os
import re
import pypdf
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pinecone import Pinecone

PDF_PATH = "/content/drive/MyDrive/voxmed_ai/8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy.pdf"
PINECONE_API_KEY = "pcsk_VTLgp_U3VRnMVNLQcoEHRD1oitC6rXmknoHVtEEkMjvx79pu1bD5Y5RHdoHD7LNnqQTWR"
INDEX_NAME = "voxmed-ai"
OHCM_EDITION = "10th"

CHAPTERS = {
    "Cardiovascular medicine": (105, 173),
    "Chest medicine":          (173, 255),
    "Gastroenterology":        (255, 305),
    "Neurology":               (457, 560),
}

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

print("Loading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="NeuML/pubmedbert-base-embeddings",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings
)

MEDICAL_SEPARATORS = ["\n\n", "\n", ". ", " "]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=MEDICAL_SEPARATORS
)

def clean_text(text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def extract_chapter_text(pdf_path, start_page, end_page):
    reader = pypdf.PdfReader(pdf_path)
    text = ""
    for i in range(start_page, min(end_page, len(reader.pages))):
        page_text = reader.pages[i].extract_text()
        if page_text:
            text += f"\n\n{page_text}"
    return text

all_chunks = []

for chapter_name, (start, end) in CHAPTERS.items():
    print(f"\nProcessing: {chapter_name} (pages {start+1}-{end})...")
    text = extract_chapter_text(PDF_PATH, start, end)
    text = clean_text(text)
    if len(text) < 100:
        print(f"  Warning: very little text")
        continue
    print(f"  Extracted {len(text):,} characters")
    chunks = splitter.split_text(text)
    print(f"  Split into {len(chunks)} chunks")
    for chunk in chunks:
        all_chunks.append(Document(
            page_content=chunk,
            metadata={
                "source": "OHCM",
                "edition": OHCM_EDITION,
                "chapter": chapter_name,
                "start_page": start + 1,
                "end_page": end,
                "type": "clinical"
            }
        ))

print(f"\nTotal chunks: {len(all_chunks)}")
print("Uploading to Pinecone...")

batch_size = 50
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    vectorstore.add_documents(batch)
    print(f"  Uploaded {min(i+batch_size, len(all_chunks))}/{len(all_chunks)} chunks")

print("\nDone! All chapters saved to Pinecone!")

Loading embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Processing: Cardiovascular medicine (pages 106-173)...
  Extracted 189,100 characters
  Split into 492 chunks

Processing: Chest medicine (pages 174-255)...
  Extracted 232,177 characters
  Split into 603 chunks

Processing: Gastroenterology (pages 256-305)...
  Extracted 165,726 characters
  Split into 427 chunks

Processing: Neurology (pages 458-560)...
  Extracted 307,751 characters
  Split into 790 chunks

Total chunks: 2312
Uploading to Pinecone...
  Uploaded 50/2312 chunks
  Uploaded 100/2312 chunks
  Uploaded 150/2312 chunks
  Uploaded 200/2312 chunks
  Uploaded 250/2312 chunks
  Uploaded 300/2312 chunks
  Uploaded 350/2312 chunks
  Uploaded 400/2312 chunks
  Uploaded 450/2312 chunks
  Uploaded 500/2312 chunks
  Uploaded 550/2312 chunks
  Uploaded 600/2312 chunks
  Uploaded 650/2312 chunks
  Uploaded 700/2312 chunks
  Uploaded 750/2312 chunks
  Uploaded 800/2312 chunks
  Uploaded 850/2312 chunks
  Uploaded 900/2312 chunks
  Uploaded 950/2312 chunks
  Uploaded 1000/2312 chunks
 